# MeterIQ — Smart Meter Dataset EDA

This notebook performs exploratory data analysis on the supplied Mathura and Bareilly smart-meter archive. It reads CSV files directly from the ZIP and does not alter raw data.

Observed raw schema: `x_Timestamp`, `t_kWh`, `z_Avg Voltage (Volt)`, `z_Avg Current (Amp)`, `y_Freq (Hz)`, `meter`. The archive also contains daily aggregated files with `meter`, `Date`, and `t_kWh`.

> Results describe data quality and potential anomalies only. They do not prove electricity theft.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

In [ ]:
# Change this path if you move the archive.
ZIP_PATH = Path('/Users/yashitarora/Downloads/archive (7).zip')

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Archive not found: {ZIP_PATH}. Update ZIP_PATH to its location.')

with ZipFile(ZIP_PATH) as archive:
    DATASETS = [item.filename for item in archive.infolist() if item.filename.lower().endswith('.csv')]

print('Available datasets:')
for dataset in DATASETS:
    print('-', dataset)

## Select and load one dataset

Start with one of the small `Aggregated` files for daily analysis. For the larger raw files, keep `SAMPLE_ROWS` set to a practical number; set it to `None` only when sufficient RAM is available.

In [ ]:
DATASET_NAME = 'SM Cleaned Data BR Aggregated.csv'  # choose an exact name printed above
SAMPLE_ROWS = 250_000  # use None to read the whole selected file

if DATASET_NAME not in DATASETS:
    raise ValueError('DATASET_NAME must be one of the archive members listed above.')

with ZipFile(ZIP_PATH) as archive:
    with archive.open(DATASET_NAME) as source:
        data = pd.read_csv(source, nrows=SAMPLE_ROWS, low_memory=False)

print(f'Loaded {len(data):,} rows from {DATASET_NAME}')
data.head()

In [ ]:
# Schema, missing values, duplicates, and observed data types
overview = pd.DataFrame({
    'dtype': data.dtypes.astype(str),
    'missing_values': data.isna().sum(),
    'missing_percent': (data.isna().mean() * 100).round(2),
    'unique_values': data.nunique(dropna=True),
})
print(f'Duplicate rows: {data.duplicated().sum():,}')
overview

In [ ]:
# Parse the actual timestamp/date column used by the chosen file.
TIME_COLUMN = 'Date' if 'Date' in data.columns else 'x_Timestamp'
CONSUMPTION_COLUMN = 't_kWh'
METER_COLUMN = 'meter'

analysis = data.copy()
analysis[TIME_COLUMN] = pd.to_datetime(analysis[TIME_COLUMN], errors='coerce')
analysis[CONSUMPTION_COLUMN] = pd.to_numeric(analysis[CONSUMPTION_COLUMN], errors='coerce')

print('Unparseable timestamps:', analysis[TIME_COLUMN].isna().sum())
print('Invalid consumption values:', analysis[CONSUMPTION_COLUMN].isna().sum())
print('Date range:', analysis[TIME_COLUMN].min(), 'to', analysis[TIME_COLUMN].max())
print('Meters:', analysis[METER_COLUMN].nunique())
analysis[[METER_COLUMN, TIME_COLUMN, CONSUMPTION_COLUMN]].head()

In [ ]:
# Numeric summary and values to investigate. Negative kWh is reported, not silently removed.
numeric_columns = analysis.select_dtypes(include='number').columns.tolist()
display(analysis[numeric_columns].describe().T)

negative_kwh = (analysis[CONSUMPTION_COLUMN] < 0).sum()
zero_kwh = (analysis[CONSUMPTION_COLUMN] == 0).sum()
q1, q3 = analysis[CONSUMPTION_COLUMN].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_flags = analysis[CONSUMPTION_COLUMN].lt(lower_bound) | analysis[CONSUMPTION_COLUMN].gt(upper_bound)
print(f'Negative kWh values: {negative_kwh:,}')
print(f'Zero kWh values: {zero_kwh:,}')
print(f'IQR values to inspect: {iqr_flags.sum():,} (bounds: {lower_bound:.4f}, {upper_bound:.4f})')

In [ ]:
# Sampling frequency is calculated separately for each meter where timestamps are available.
ordered = analysis.dropna(subset=[METER_COLUMN, TIME_COLUMN]).sort_values([METER_COLUMN, TIME_COLUMN])
intervals = ordered.groupby(METER_COLUMN)[TIME_COLUMN].diff().dropna()
if intervals.empty:
    print('Not enough timestamped readings to estimate sampling frequency.')
else:
    print('Median reading interval:', intervals.median())
    print('Most common reading intervals:')
    display(intervals.value_counts().head(10))

In [ ]:
# Meter-level consumption profile
meter_summary = (analysis.groupby(METER_COLUMN)[CONSUMPTION_COLUMN]
                 .agg(records='size', total_kwh='sum', mean_kwh='mean', min_kwh='min', max_kwh='max')
                 .sort_values('total_kwh', ascending=False))
meter_summary.head(15)

In [ ]:
# Plot the selected meter. This is a visual inspection aid, not a theft classification.
SELECTED_METER = meter_summary.index[0]
meter_data = analysis.loc[analysis[METER_COLUMN] == SELECTED_METER].sort_values(TIME_COLUMN)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(meter_data[TIME_COLUMN], meter_data[CONSUMPTION_COLUMN], linewidth=0.8, label='Consumption (kWh)')
flagged = meter_data.loc[iqr_flags.reindex(meter_data.index, fill_value=False)]
ax.scatter(flagged[TIME_COLUMN], flagged[CONSUMPTION_COLUMN], color='crimson', s=12, label='IQR value to inspect')
ax.set(title=f'Consumption history — {SELECTED_METER}', xlabel='Time', ylabel='kWh')
ax.legend()
plt.show()

## Next steps

Verify the timestamp, meter, and consumption fields against the source documentation before cleaning or training. Decide how invalid readings and missing intervals should be handled after inspecting the outputs above. The provided files contain no observed verified theft-label column, so anomaly flags should be presented as potential anomalies—not proven theft.